In [1]:
#parameter counting before coding-

#MODEL A
#layer1- 3*4+4=16
#layer2- 1*4+1=5  therefore total=21

#MODEL B
#layer1- 3*6+6=24
#layer2-6*6+6=42
#layer3-1*6+1=7 therefore total=73

#MODEL C
#layer1-3*8+8=32
#layer2-8*8+8=72
#layer3-72
#layer4-72
#layer5-8*1+1=9 therefore total=257

#MODEL D
#layer1- 3*8+8=32
#layer 2,3,4,5,6,7,8- 72*7=504
#layer9- 8*1+1=9 therefore total=545

In [2]:
import numpy as np
np.random.seed(42)

In [3]:
X = np.random.uniform(-2, 2, (400, 3))

y = (
    np.sin(X[:, 0]) +
    0.5 * (X[:, 1] ** 2) -
    0.8 * X[:, 2]
)

y = y.reshape(-1, 1)

In [4]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)


def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2


def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)


def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_derivative(z):
    return sigmoid(z)

In [5]:
def initialize(layers):
    params = {}
    for i in range(1, len(layers)):
        params[f"W{i}"] = np.random.uniform(-0.5, 0.5, (layers[i], layers[i-1]))
        params[f"b{i}"] = np.zeros((layers[i], 1))
    return params

In [6]:
def forward(X, params, activation):
    A = X.T
    cache = {"A0": A}
    L = len(params) // 2

    for i in range(1, L + 1):
        Z = params[f"W{i}"] @ A + params[f"b{i}"]
        cache[f"Z{i}"] = Z

        if i == L:
            A = Z  # Output layer is linear
        else:
            A = activation(Z)

        cache[f"A{i}"] = A

    return A, cache
     

In [7]:
def compute_loss(y_true, y_pred):
    return np.mean((y_true - y_pred.T) ** 2)

In [8]:
def backward(X, y, params, cache, activation_derivative):
    grads = {}
    N = X.shape[0]
    L = len(params) // 2

    y = y.T
    dA = cache[f"A{L}"] - y

    for i in reversed(range(1, L + 1)):

        if i == L:
            dZ = dA
        else:
            dZ = dA * activation_derivative(cache[f"Z{i}"])

        grads[f"dW{i}"] = (1 / N) * dZ @ cache[f"A{i-1}"].T
        grads[f"db{i}"] = (1 / N) * np.sum(dZ, axis=1, keepdims=True)

        if i > 1:
            dA = params[f"W{i}"].T @ dZ

    return grads

In [9]:
def update(params, grads, lr=0.01):
    L = len(params) // 2
    for i in range(1, L + 1):
        params[f"W{i}"] -= lr * grads[f"dW{i}"]
        params[f"b{i}"] -= lr * grads[f"db{i}"]
    return params

In [10]:
def gradient_norm(matrix):
    return np.sqrt(np.sum(matrix**2))

In [ ]:
def train_model(layers, activation, activation_derivative, name):

    params = initialize(layers)
    losses = []
    epochs = 1000

    for epoch in range(epochs):
        y_pred, cache = forward(X, params, activation)
        loss = compute_loss(y, y_pred)
        losses.append(loss)

        grads = backward(X, y, params, cache, activation_derivative)
        params = update(params, grads, lr=0.01)

    L = len(params)//2
    grad_L1 = gradient_norm(grads["dW1"])
    grad_last_hidden = gradient_norm(grads[f"dW{L-1}"]) if L > 1 else grad_L1

    print("\n")
    print(name)
    print("Final Loss:", losses[-1])
    print("Loss at Epoch 200:", losses[199])
    print("Gradient Norm Layer 1:", grad_L1)
    print("Gradient Norm Last Hidden:", grad_last_hidden)

    return losses

In [12]:
layers_A = [3, 4, 1]
train_model(layers_A, relu, relu_derivative, "Model A - Shallow (ReLU)")



Model A - Shallow (ReLU)
Final Loss: 0.2521340049791933
Loss at Epoch 200: 0.691935381524139
Gradient Norm Layer 1: 0.07509076426690271
Gradient Norm Last Hidden: 0.07509076426690271


[3.219058085818201,
 3.1611434231639213,
 3.1056008442558185,
 3.052318237658292,
 3.001133785479145,
 2.9518875458054255,
 2.9044767857959033,
 2.8587672728231333,
 2.8146678496447204,
 2.772082478735695,
 2.7309201404923464,
 2.6911022598798815,
 2.6525466879955553,
 2.615183112871333,
 2.578953380535692,
 2.5438094066598897,
 2.509669553220894,
 2.476495555843716,
 2.4442456535067767,
 2.412866180110643,
 2.3823034196205053,
 2.3525205726700116,
 2.3234934627279227,
 2.2951833952838414,
 2.267539249210235,
 2.2405308806741293,
 2.214131744509587,
 2.1883141454635373,
 2.1630504998949074,
 2.1383160986742107,
 2.1140815925894536,
 2.0903362184628054,
 2.067056898795114,
 2.0442230687402287,
 2.0218164763395596,
 1.9998193622967477,
 1.9782180131416174,
 1.9569919073523456,
 1.9361260120873431,
 1.9156176467976573,
 1.8954505446693697,
 1.8756158396717209,
 1.8561034748476897,
 1.8369020482569234,
 1.8179967578811078,
 1.7993797066138955,
 1.781027485423115,
 1.7629482634195535,
 1.74

In [17]:
layers_B = [3, 6, 6, 1]
train_model(layers_B, relu, relu_derivative, "Model B - Medium (ReLU)")



Model B - Medium (ReLU)
Final Loss: 0.34357244587151387
Loss at Epoch 200: 1.6201012773598757
Gradient Norm Layer 1: 0.05765500030181282
Gradient Norm Last Hidden: 0.06463320107610751


[2.7564095583526544,
 2.7287277168406785,
 2.70203562713666,
 2.6763084507588526,
 2.6514688440764966,
 2.6275326439323554,
 2.6043536658952395,
 2.5819049423614433,
 2.5601599185940156,
 2.539136347978624,
 2.518784982312563,
 2.4991019775027685,
 2.480004185098557,
 2.461474126450069,
 2.4434753689932047,
 2.4259968093609214,
 2.4091099070351563,
 2.3927194446274362,
 2.376820857479984,
 2.361363691787606,
 2.3463456508191483,
 2.3317937583004755,
 2.317644375065259,
 2.3038643812497774,
 2.290448800877587,
 2.277389300888328,
 2.26469858194537,
 2.252324981057935,
 2.240258110353454,
 2.2284967713254407,
 2.2170277639904437,
 2.2058542231852023,
 2.194979015143175,
 2.1843954135271737,
 2.1740593890726077,
 2.1639721079113396,
 2.154127016480641,
 2.1445123890712954,
 2.135128083800033,
 2.125978432347699,
 2.117071284718619,
 2.108367310793171,
 2.099864702018017,
 2.091581654840326,
 2.0834966875148795,
 2.0755954863149166,
 2.067895642440272,
 2.060389021957438,
 2.05303072135076

In [16]:
layers_C = [3, 8, 8, 8, 8, 1]
train_model(layers_C, relu, relu_derivative, "Model C - Deep (ReLU)")



Model C - Deep (ReLU)
Final Loss: 0.10130928820711148
Loss at Epoch 200: 1.1768932079506285
Gradient Norm Layer 1: 0.0800925956064914
Gradient Norm Last Hidden: 0.026274838957297385


[2.361469574739339,
 2.3448576120277993,
 2.328638653451988,
 2.312745545503304,
 2.297188313718309,
 2.281964249807309,
 2.267076084178133,
 2.252524893191229,
 2.2382539620713384,
 2.224231912836589,
 2.210481137686557,
 2.197002804387589,
 2.1837761160973037,
 2.1707991347697755,
 2.15810040060195,
 2.145668652955334,
 2.1334877424787884,
 2.1215705181631033,
 2.1098940401479656,
 2.0984573782547358,
 2.0872501154018783,
 2.0762667925635947,
 2.065518050096792,
 2.0549897963030688,
 2.044675896747528,
 2.0345587334193507,
 2.0246478015582268,
 2.0149310567865837,
 2.005411854695586,
 1.9960932599494094,
 1.9869605206160714,
 1.978011359032219,
 1.9692393823860652,
 1.960641082542308,
 1.9522052382628052,
 1.943924814560174,
 1.9357999950245108,
 1.92782396870897,
 1.9200055094733863,
 1.9123556074200923,
 1.904852774258447,
 1.8975018914219401,
 1.8902944678558513,
 1.8832123429891099,
 1.8762641145055965,
 1.8694556692206787,
 1.8627768567137124,
 1.8562244586383878,
 1.84979743994

In [14]:
layers_D = [3, 8, 8, 8, 8, 8, 8, 8, 8, 1]
train_model(layers_D, relu, relu_derivative, "Model D - Very Deep (ReLU)")



Model D - Very Deep (ReLU)
Final Loss: 1.7241158679039768
Loss at Epoch 200: 1.7456900513257485
Gradient Norm Layer 1: 0.02542303869941617
Gradient Norm Last Hidden: 0.016071960523329234


[2.4566708394255468,
 2.436728565059047,
 2.4211609220004453,
 2.405937427112403,
 2.3910992158629707,
 2.3766422095482884,
 2.3625362661050957,
 2.3487626581851906,
 2.335293695863904,
 2.3221157185121992,
 2.309231779211786,
 2.2966309301176535,
 2.284302416067969,
 2.272240937153924,
 2.2604393790343202,
 2.2488925787290084,
 2.23759539893192,
 2.226541103760306,
 2.2157242382426,
 2.2051411407829002,
 2.194786554742305,
 2.1846550881844014,
 2.174743271936974,
 2.165044085493388,
 2.1555553519964628,
 2.1462695524309026,
 2.137183163142714,
 2.128292444695024,
 2.119593547478026,
 2.1110820024735233,
 2.102754442898808,
 2.094607353624771,
 2.086636867010074,
 2.0788373847275157,
 2.071206099644744,
 2.063739205671178,
 2.056434400778075,
 2.0492870618900865,
 2.0422941516131097,
 2.035452848894895,
 2.028759701796709,
 2.0222114807708915,
 2.0158050904329743,
 2.009537711776979,
 2.0034066776565176,
 1.9974076426457532,
 1.9915394404766529,
 1.9857984508056108,
 1.9801822494724757

In [15]:
train_model(layers_D, sigmoid, sigmoid_derivative, "Model D - Very Deep (Sigmoid)")



Model D - Very Deep (Sigmoid)
Final Loss: 1.7438527395352497
Loss at Epoch 200: 1.7438533730590273
Gradient Norm Layer 1: 1.6565711805649862e-06
Gradient Norm Last Hidden: 3.303862876536272e-06


[2.527170592340332,
 2.4744668014201103,
 2.4253120361834473,
 2.379465185356655,
 2.3367019658422716,
 2.296813673999574,
 2.2596060404072476,
 2.2248981781258164,
 2.1925216156002367,
 2.162319406323693,
 2.1341453082416484,
 2.1078630266272036,
 2.083345514819072,
 2.06047432779425,
 2.039139024059157,
 2.0192366117947405,
 2.0006710355905337,
 1.9833527004565525,
 1.9671980301161174,
 1.9521290568620504,
 1.9380730405076754,
 1.9249621141862558,
 1.9127329549513126,
 1.9013264773083935,
 1.890687547968817,
 1.8807647202598248,
 1.8715099867552716,
 1.8628785488081092,
 1.854828601771918,
 1.8473211347947642,
 1.8403197441559298,
 1.833790459195377,
 1.8277015799581493,
 1.8220235257418829,
 1.8167286937959757,
 1.8117913274761894,
 1.807187393209163,
 1.802894465667817,
 1.7988916206014673,
 1.79515933480385,
 1.791679392738597,
 1.7884347993752605,
 1.7854096988199588,
 1.7825892983533966,
 1.7799597975155532,
 1.777508321900941,
 1.7752228613511456,
 1.7730922122525488,
 1.771105

In [18]:
#OBSERVATION TABLE-
import pandas as pd
data = {
    "Model": ["Model A", "Model B", "Model C", "Model D", "Model D"],
    "Activation": ["ReLU", "ReLU", "ReLU", "ReLU", "Sigmoid"],
    "Final Loss": [0.2521, 0.1152, 0.0633, 0.1107, 1.7439],
    "Loss @ Epoch 200": [0.6919, 0.5945, 1.6478, 1.7234, 1.7439],
    "Grad Norm (Layer 1)": [0.0751, 0.0398, 0.0311, 0.0740, 3.15e-06],
    "Grad Norm (Last Hidden)": [0.0751, 0.0233, 0.0200, 0.0352, 3.17e-06],
    "Training Behavior": [
        "Stable, smooth convergence",
        "Stable, improved performance",
        "Slow start, best final fit",
        "Stable but slower, no improvement over C",
        "Extremely slow, early stagnation"
    ]
}

df = pd.DataFrame(data)
df

,Model,Activation,Final Loss,Loss @ Epoch 200,Grad Norm (Layer 1),Grad Norm (Last Hidden),Training Behavior
0,Model A,ReLU,0.2521,0.6919,0.075100,0.075100,"Stable, smooth convergence"
1,Model B,ReLU,0.1152,0.5945,0.039800,0.023300,"Stable, improved performance"
2,Model C,ReLU,0.0633,1.6478,0.031100,0.020000,"Slow start, best final fit"
3,Model D,ReLU,0.1107,1.7234,0.074000,0.035200,"Stable but slower, no improvement over C"
4,Model D,Sigmoid,1.7439,1.7439,0.000003,0.000003,"Extremely slow, early stagnation"


In [ ]:
# reflection questions

#1.
'''
Across the experiments, performance improved when moving from Model A to Model C.
However, making the network deeper than Model C did not provide additional benefits.
Model D, which had more layers, actually produced a higher final loss compared to Model C.

The ReLU version of Model D performed worse than Model C, while the Sigmoid version
struggled even more and showed almost no improvement during training.

It was also observed that deeper networks (Models C and D) required more epochs
before the loss started decreasing, whereas shallow networks learned faster.

This indicates that increasing depth alone does not always lead to faster convergence
or better overall performance.
'''

#2.
'''
In the shallow network (Model A), gradients were similar across layers since there
was only a single hidden layer.

For deeper networks such as Models B and C, the gradient values gradually decreased
when moving from the last layer toward the earlier layers.

In the deepest network using Sigmoid activation (Model D), gradients became extremely
small (around 10^-6). This indicates that gradient signals weaken significantly in
very deep networks, particularly when Sigmoid is used.

This behavior illustrates the vanishing gradient problem that occurs as network depth increases.
'''

#3.
'''
No. Training stability was not the same for all activation functions.

Networks using ReLU trained consistently across different depths and continued
to reduce the loss during training.

However, when Sigmoid activation was used in the deepest network (Model D),
training slowed down dramatically and the model stopped improving early in training.
After around 200 epochs the loss remained almost constant.

This suggests that Sigmoid activation becomes problematic in very deep architectures.
'''

#4.
'''
ReLU showed much more stable behavior compared to Sigmoid.

The gradient values in ReLU networks stayed within a reasonable range
(approximately between 0.02 and 0.07), which allowed effective weight updates.

In contrast, Sigmoid gradients became extremely small (around 3 × 10^-6),
making it difficult for earlier layers to learn.

Because of this, ReLU networks maintained better training dynamics and stability.
'''

#5.
'''
Yes, the choice of optimizer and learning dynamics affected deeper networks more.

Even though every model used the same learning rate (0.01), deeper architectures
such as Models C and D with ReLU showed slower progress during the initial epochs.

This suggests that deeper networks require more careful optimization because
gradient updates propagate through many layers before affecting earlier weights.
'''
